# Week 5 Assignment - Apache Spark Fundamentals

## Name: Ayushi Gupta
### Objective
To understand Apache Spark fundamentals and perform data cleaning, transformation, filtering, aggregation, and grouping operations using Spark DataFrames.


#Q.1 What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

Answer:

* MapReduce stores intermediate results on disk after each stage.
* High disk I/O causes slower execution.
* Not suitable for iterative machine learning workloads.
* Requires complex code for simple operations.
* Higher latency for interactive analytics.

Spark Advantages:

* In-memory processing.
* Faster execution.
* Easier APIs (DataFrames, SQL).
* Supports machine learning, streaming, and graph processing.

#Q.2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

Spark stores intermediate datasets in RAM instead of writing them repeatedly to disk.

Benefits:

Faster data access.
Reduced disk operations.
Ideal for iterative machine learning algorithms where the same data is processed multiple times.

Example:

A machine learning model requiring 100 iterations can reuse cached data in memory rather than reading from disk every iteration.

#Install PySpark

In [ ]:
!pip install pyspark

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week5Assignment") \
    .getOrCreate()

#Load Dataset

In [ ]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv("superstore_raw.csv")

In [ ]:
df.show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|           City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|       Claire Gute|   Consumer|United States|      Henderson|      Kentucky|      42420|  South|FUR-BO-10001798|   

In [ ]:
df.cache()

DataFrame[Row ID: int, Order ID: string, Order Date: string, Ship Date: string, Ship Mode: string, Customer ID: string, Customer Name: string, Segment: string, Country: string, City: string, State: string, Postal Code: int, Region: string, Product ID: string, Category: string, Sub-Category: string, Product Name: string, Sales: double, Quantity: int, Discount: double, Profit: double]

In [ ]:
print("Rows:", df.count())

Rows: 9994


#Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.

In [ ]:
df_no_duplicates = df.dropDuplicates()

print("Original Row Count:", df.count())
print("Row Count After Removing Duplicates:", df_no_duplicates.count())

df_no_duplicates.show(5)

Original Row Count: 9994
Row Count After Removing Duplicates: 9994
+------+--------------+----------+---------+--------------+-----------+----------------+---------+-------------+------------+------------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+----------+
|Row ID|      Order ID|Order Date|Ship Date|     Ship Mode|Customer ID|   Customer Name|  Segment|      Country|        City|       State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|    Profit|
+------+--------------+----------+---------+--------------+-----------+----------------+---------+-------------+------------+------------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+----------+
|    28|US-2015-150630| 9/17/2015|9/21/2015|Standard Class|   TB-21520| Tracy Blumstein| Consumer|United States|Philadelphia|Pennsylvania|      19140|

The dropDuplicates() function was used to remove duplicate records from the dataset. This helps ensure data quality and prevents duplicate entries from affecting analysis results.

#Q4: Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.

In [ ]:
from pyspark.sql.functions import avg, col

df_no_duplicates.filter(
    col("Region") == "West"
).groupBy(
    "Category"
).agg(
    avg("Sales").alias("Average_Sales")
).show()

+---------------+------------------+
|       Category|     Average_Sales|
+---------------+------------------+
|Office Supplies|116.42237691091195|
|      Furniture| 357.3023246110331|
|     Technology| 420.6875325542569|
+---------------+------------------+



The dataset was filtered for the West region and grouped by Category. The average sales for each category were then calculated.

In [ ]:
# Count null values before cleaning
print("Null Customer Names:",
      df_no_duplicates.filter(col("Customer Name").isNull()).count())

print("Null Ship Modes:",
      df_no_duplicates.filter(col("Ship Mode").isNull()).count())

Null Customer Names: 0
Null Ship Modes: 0


#Q5: What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.

In [ ]:
# Using .na.drop() to remove rows where Customer Name is null
df_drop = df_no_duplicates.na.drop(subset=["Customer Name"])

print("Rows after dropping null Customer Names:")
print(df_drop.count())

# Using .na.fill() to replace null values in Ship Mode
df_fill = df_no_duplicates.na.fill({
    "Ship Mode": "Unknown"
})

df_fill.select("Ship Mode").show(5)

Rows after dropping null Customer Names:
9994
+--------------+
|     Ship Mode|
+--------------+
|Standard Class|
|   First Class|
|  Second Class|
|Standard Class|
|  Second Class|
+--------------+
only showing top 5 rows


.na.drop() removes rows containing null values.

.na.fill() replaces null values with a specified value while retaining the rows.

Here, missing values in Ship Mode were replaced with "Unknown".

No null values were found in the dataset. Therefore, applying .na.drop() and .na.fill() did not alter the data, but the functions were demonstrated as part of the data cleaning process.

#Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

In [ ]:
from pyspark.sql.functions import count

df_no_duplicates.groupBy(
    "City"
).agg(
    count("*").alias("Total_Orders")
).filter(
    col("Total_Orders") > 100
).show()

+-------------+------------+
|         City|Total_Orders|
+-------------+------------+
|  Springfield|         163|
|       Dallas|         157|
| Philadelphia|         537|
|  Los Angeles|         747|
|San Francisco|         510|
|    San Diego|         170|
|      Detroit|         115|
|     Columbus|         222|
|      Chicago|         314|
|      Seattle|         428|
|New York City|         915|
|      Houston|         377|
| Jacksonville|         125|
+-------------+------------+



# Q7: How does the immutability of Spark DataFrames affect data cleaning operations?
Answer:

Spark DataFrames are immutable, which means their data cannot be modified after they are created. Any data cleaning operation such as dropping columns, renaming columns, filtering rows, or handling null values does not change the original DataFrame. Instead, Spark creates and returns a new DataFrame containing the updated data.

This approach improves reliability, fault tolerance, and enables Spark to optimize execution through its query planner.

In [ ]:
# Dropping a column
new_df = df.drop("Postal Code")

# Renaming a column
renamed_df = df.withColumnRenamed("Customer Name", "Customer_Name")

#Q8: Write a Spark command to filter a dataset for rows where the region is "west" and the Segment is equal to "Consumer".

In [ ]:
from pyspark.sql.functions import col

df_no_duplicates.filter(
    (col("Region") == "West") &
    (col("Segment") == "Consumer")
).show()

+------+--------------+----------+----------+--------------+-----------+-------------------+--------+-------------+----------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|      Customer Name| Segment|      Country|            City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+-------------------+--------+-------------+----------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|   203|CA-2014-133690|  8/3/2014|  8/5/2014|   First Class|   BS-11755|      Bruce Stewart|Consumer|United States|          Denver|  Colorado|      80219|  West|OFF-AP-10003622|Office Supplies|  Appliances|B

The original question required filtering records where age is between 18 and 30 and subscription is 'Premium'. Since the Superstore dataset does not contain Age and Subscription columns, an equivalent filtering operation was performed using Region and Segment. The dataset was filtered to display records belonging to the West region and Consumer segment.
* from pyspark.sql.functions import col
from pyspark.sql.functions import col

df.filter(
    (col("age").between(18, 30)) &
    (col("subscription") == "Premium")
).show()

# Q9: When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?
Answer:

Null values should be handled before performing aggregations such as sum(), avg(), min(), or max() because they can lead to inaccurate or misleading analytical results. Missing values may affect calculations, reduce the number of records considered during aggregation, or produce unexpected outputs.

Cleaning null values beforehand ensures that the results are reliable, consistent, and representative of the actual dataset. Common approaches include removing records with missing values using .na.drop() or replacing them using .na.fill().

#Q10: Write the code to revise a column named Order_Date by casting it to a TimestampType and renaming it to event_time.

In [ ]:
from pyspark.sql.functions import to_timestamp, col

df_time = df_no_duplicates.withColumn(
    "Order Date",
    to_timestamp(col("Order Date"), "M/d/yyyy")
).withColumnRenamed(
    "Order Date",
    "event_time"
)

df_time.show(5, False)

+------+--------------+-------------------+---------+--------------+-----------+----------------+---------+-------------+------------+------------+-----------+------+---------------+---------------+------------+-------------------------------------------------------------------------------+-------+--------+--------+----------+
|Row ID|Order ID      |event_time         |Ship Date|Ship Mode     |Customer ID|Customer Name   |Segment  |Country      |City        |State       |Postal Code|Region|Product ID     |Category       |Sub-Category|Product Name                                                                   |Sales  |Quantity|Discount|Profit    |
+------+--------------+-------------------+---------+--------------+-----------+----------------+---------+-------------+------------+------------+-----------+------+---------------+---------------+------------+-------------------------------------------------------------------------------+-------+--------+--------+----------+
|28    |US-20

The original question required converting a column named `raw_timestamp` to `TimestampType` and renaming it to `event_time`.

Since the Superstore dataset contains dates in the `Order Date` column with the format M/d/yyyy, the `to_timestamp()` function was used to convert the string values into Spark's TimestampType format. The resulting column was named `event_time`.

# Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?
Answer

Shuffle is the process of redistributing data across different partitions during operations such as groupBy(), join(), distinct(), and reduceByKey().

During a grouping operation, Spark moves records with the same key to the same partition so that calculations can be performed correctly. This movement of data between worker nodes is called a shuffle.

A shuffle is considered a wide transformation because data must move across partitions rather than being processed independently within the existing partition. Since network communication and disk I/O may be involved, shuffle operations are generally more expensive and can impact performance.

#Q12: Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.

In [ ]:
from pyspark.sql.functions import col

clean_df = df_no_duplicates.filter(
    col("Customer Name").isNotNull() &
    (col("Customer Name") != "")
)

clean_df.show(5)

+------+--------------+----------+---------+--------------+-----------+----------------+---------+-------------+------------+------------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+----------+
|Row ID|      Order ID|Order Date|Ship Date|     Ship Mode|Customer ID|   Customer Name|  Segment|      Country|        City|       State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|    Profit|
+------+--------------+----------+---------+--------------+-----------+----------------+---------+-------------+------------+------------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+----------+
|    28|US-2015-150630| 9/17/2015|9/21/2015|Standard Class|   TB-21520| Tracy Blumstein| Consumer|United States|Philadelphia|Pennsylvania|      19140|  East|FUR-BO-10004834|      Furniture|   Bookcases|Riverside Palai

The original question required removing rows where the email column contained null values or the username column was empty.

Since the Superstore dataset does not contain email or username columns, the Customer Name column was used as an equivalent field. Records with null or empty customer names were filtered out to improve data quality.

#Q13: How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?

In [ ]:
from pyspark.sql.functions import min, max, avg

df_no_duplicates.agg(
    min("Sales").alias("Minimum_Sales"),
    max("Sales").alias("Maximum_Sales"),
    avg("Sales").alias("Average_Sales")
).show()

+-------------+-------------+-----------------+
|Minimum_Sales|Maximum_Sales|    Average_Sales|
+-------------+-------------+-----------------+
|        0.444|     22638.48|229.8580008304968|
+-------------+-------------+-----------------+



The .agg() function allows multiple aggregation operations to be performed in a single statement.

In this example:
- min() calculates the minimum sales value.
- max() calculates the maximum sales value.
- avg() calculates the average sales value.

This provides multiple statistics efficiently in one operation.

#Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?
Answer:
When inferSchema=True is used on datasets containing inconsistent or messy date formats, Spark may incorrectly identify the data type of a column.

For example, some values may be interpreted as dates while others may be treated as strings or null values. This can lead to schema inconsistencies, conversion errors, and incorrect analytical results.

Therefore, it is often better to explicitly define schemas or standardize date formats before loading the data.

#Q15: Write a final processing pipeline that:

1.Filters out duplicates.

2.Fills null prices with 0.

3.Groups by store_id to calculate total revenue.

In [ ]:
from pyspark.sql.functions import sum

pipeline_df = (
    df
    .dropDuplicates()
    .na.fill({"Sales": 0})
    .groupBy("Region")
    .agg(
        sum("Sales").alias("Total_Revenue")
    )
)

pipeline_df.show()

+-------+-----------------+
| Region|    Total_Revenue|
+-------+-----------------+
|  South|       391721.905|
|Central|501239.8907999995|
|   East|678781.2400000005|
|   West|725457.8245000001|
+-------+-----------------+



A complete data processing pipeline was created by combining multiple Spark transformations.

Steps performed:
1. Removed duplicate records using dropDuplicates().
2. Replaced missing Sales values with 0 using .na.fill().
3. Grouped data by Region.
4. Calculated total revenue using sum(Sales).

This demonstrates how Spark transformations can be chained together to build efficient data processing workflows.

In [ ]:
final_df = (
    df
    .dropDuplicates()
    .na.fill({"Ship Mode": "Unknown"})
    .filter(col("Customer Name").isNotNull())
    .withColumn(
        "event_time",
        to_timestamp(col("Order Date"), "M/d/yyyy")
    )
)

In [ ]:
final_df.toPandas().to_csv("results.csv", index=False)

# Conclusion

In this assignment, Apache Spark DataFrames were used to perform data cleaning, filtering, transformation, aggregation, and grouping operations. The assignment demonstrated Spark's advantages over MapReduce, including in-memory processing, faster execution, and ease of use for large-scale data analysis.

## Brief Insights

1. Apache Spark performed data processing efficiently using DataFrames and in-memory computation, making operations faster compared to traditional MapReduce.

2. Duplicate records were successfully identified and removed using the `dropDuplicates()` function, improving data quality.

3. The dataset was checked for missing values and data cleaning techniques such as `.na.drop()` and `.na.fill()` were demonstrated.

4. Filtering operations helped extract specific subsets of data based on business conditions such as Region and Segment.

5. Data transformation techniques, including column renaming and timestamp conversion, improved the usability and consistency of the dataset.

6. Aggregation functions such as `count()`, `avg()`, `min()`, and `max()` provided useful statistical summaries of sales data.

7. GroupBy operations revealed meaningful insights by organizing data into categories and regions for analysis.

8. The final processing pipeline demonstrated how multiple Spark transformations can be combined to clean, transform, and analyze data efficiently in a single workflow.

9. The assignment highlighted the importance of schema management, data cleaning, and understanding shuffle operations for scalable big data processing.

10. Overall, Spark DataFrames provided a simple and powerful approach for handling large datasets and performing analytical operations efficiently.
